# Recommender Evaluation Workflow

This notebook outlines an offline evaluation loop for the profile playlist recommender. The goal is to tune scoring weights and candidate generation using saved viewing history instead of relying only on manual inspection.

The evaluation uses a temporal holdout: train on older viewing events, generate candidate recommendations, then check whether the held-out later period contains overlapping titles or similar metadata signals.

## Metrics To Track

- `hit_rate@k`: whether at least one held-out title appears in the top K recommendations.
- `recall@k`: share of held-out titles recovered in the top K recommendations.
- `genre_coverage@k`: how many preferred held-out genres are represented.
- `language_country_match@k`: how often recommendations match held-out language or origin-country preferences.
- `novelty@k`: penalty for over-recommending globally popular titles.
- `diversity@k`: spread across genre, media type, language, country, and release decade.

Exact title hits will usually be low for Netflix history because many watched titles may not map cleanly to TMDB, and TMDB recommendations are not Netflix availability-aware. Treat the metadata metrics as equally important.

In [ ]:
import os
import sys
from collections import Counter
from datetime import timedelta
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "mysite.settings")

import django
django.setup()

from django.db.models import Max

from api.models import NetflixProfile, ViewingEvent
from api.services import recommendations as recs

In [ ]:
EVAL_K = 20
HOLDOUT_DAYS = 90

def profile_windows(profile, holdout_days=HOLDOUT_DAYS):
    latest = ViewingEvent.objects.filter(profile=profile).aggregate(latest=Max("started_at"))["latest"]
    if latest is None:
        return None, None
    holdout_start = latest.date() - timedelta(days=holdout_days - 1)
    train_events = ViewingEvent.objects.filter(
        profile=profile,
        title__isnull=False,
        started_at__date__lt=holdout_start,
    ).select_related("title")
    holdout_events = ViewingEvent.objects.filter(
        profile=profile,
        title__isnull=False,
        started_at__date__gte=holdout_start,
    ).select_related("title")
    return list(train_events), list(holdout_events)

def title_key(title):
    media_type = recs._media_type_for_title(title)
    if title.tmdb_id and media_type:
        return (media_type, str(title.tmdb_id))
    return ("name", (title.canonical_name or title.name).strip().casefold())

In [ ]:
def weighted_rows_from_events(events, period_end):
    by_title = {}
    for event in events:
        row = by_title.setdefault(
            event.title_id,
            {"title": event.title, "weight": 0.0, "seconds": 0, "latest": event.started_at},
        )
        age_days = max(0, (period_end - event.started_at.date()).days)
        recency = recs.exp(-recs.log(2) * age_days / 45)
        row["seconds"] += event.duration_seconds
        row["weight"] += max(event.duration_seconds, 60) * recency
        row["latest"] = max(row["latest"], event.started_at)
    return sorted(by_title.values(), key=lambda row: row["weight"], reverse=True)

def evaluate_profile(profile, k=EVAL_K):
    train_events, holdout_events = profile_windows(profile)
    if not train_events or not holdout_events:
        return None

    period_end = max(event.started_at.date() for event in train_events)
    weighted_titles = weighted_rows_from_events(train_events, period_end)
    preferences = recs._profile_preferences(weighted_titles)
    hyperparameters = recs._generate_hyperparameters(weighted_titles)
    candidates = recs._discover_candidates(weighted_titles, preferences, hyperparameters)
    scored = recs._score_candidates(weighted_titles, candidates, preferences, hyperparameters)
    ranked = recs._diversify(scored, hyperparameters)[:k]

    heldout_keys = {title_key(event.title) for event in holdout_events}
    recommended_keys = {(candidate.media_type, str(candidate.external_id)) for candidate, _, _ in ranked}
    hits = heldout_keys & recommended_keys

    heldout_genres = Counter(
        genre for event in holdout_events for genre in (event.title.genres or [])
    )
    recommended_genres = Counter(
        genre for candidate, _, _ in ranked for genre in (candidate.genres or [])
    )
    heldout_languages = Counter(event.title.original_language for event in holdout_events if event.title.original_language)
    recommended_languages = Counter(candidate.original_language for candidate, _, _ in ranked if candidate.original_language)

    return {
        "profile": profile.name,
        "train_titles": len({event.title_id for event in train_events}),
        "holdout_titles": len(heldout_keys),
        "candidate_count": len(candidates),
        "hit_rate_at_k": 1.0 if hits else 0.0,
        "recall_at_k": round(len(hits) / max(1, len(heldout_keys)), 4),
        "genre_coverage_at_k": round(
            len(set(heldout_genres) & set(recommended_genres)) / max(1, len(set(heldout_genres))),
            4,
        ),
        "language_overlap_at_k": round(
            len(set(heldout_languages) & set(recommended_languages)) / max(1, len(set(heldout_languages))),
            4,
        ),
        "metadata_richness": hyperparameters["metadata_richness"],
        "score_weights": hyperparameters["score_weights"],
    }

In [ ]:
rows = []
for profile in NetflixProfile.objects.all().order_by("name"):
    result = evaluate_profile(profile)
    if result:
        rows.append(result)

rows

## Hyperparameter Experiments

Use the same temporal split to compare alternate scoring weights. Keep TMDB call limits low and prefer cached `ExternalCatalogTitle` rows for repeated experiments.

In [ ]:
WEIGHT_EXPERIMENTS = [
    {"name": "baseline", "content_similarity": 0.46, "genre_affinity": 0.18},
    {"name": "genre_heavy", "content_similarity": 0.36, "genre_affinity": 0.28},
    {"name": "metadata_light", "content_similarity": 0.30, "genre_affinity": 0.22},
]

def apply_weight_override(hyperparameters, override):
    next_params = {**hyperparameters, "score_weights": dict(hyperparameters["score_weights"])}
    for key, value in override.items():
        if key != "name":
            next_params["score_weights"][key] = value
    total = sum(next_params["score_weights"].values())
    next_params["score_weights"] = {
        key: round(value / total, 4)
        for key, value in next_params["score_weights"].items()
    }
    return next_params

## Interpreting Results

A low exact-title hit rate is expected unless the catalog cache contains many titles that later appear in the user's history. Stronger signals for this app are genre coverage, language/country overlap, and whether recommendations feel coherent without collapsing into the same genre or title type.

Before changing production weights, compare at least three profiles and inspect the actual top recommendations for each experiment.